Q1
(a): None
문자열의 시작이 숫자가 아니기 때문입니다.
(b): 2026-05-06
문자열 전체에서 숫자 4개, 하이픈, 숫자 2개, 숫자 2개와 첫 번째로 매칭되는 부분입니다.
(c): ['2026-05-06', '2026-05-18']
일치하는 모든 문자열을 리스트로.
(d): [('2026', '05', ,06,), ('2026', '05', '18')]
캡쳐 그룹이 있으면 그룹만 튜플리스트로.
(e): ['2026-05-06', '2026-05-18']
?:가 있으면 값을 캡쳐하지 않습니다.

추가질문: (c)와 (e)는 캡처 그룹이 없거나 비캡처 그룹을 사용하여 일치하는 전체 문자열을 반환하는 반면, (d)는 괄호로 지정된 각 그룹의 매칭 결과만을 튜플 형태로 추출하여 반환하기 때문입니다.

Q2
(a): '[T]!'
첫 번째 <부터 마지막 >까지 모든 내용을 치환합니다.
(b): '[T]안녕[T] [T]세상[T]!'
각 <> 안 최소한만 치환합니다.
(c): '[T]안녕[T] [T]세상[T]!'
> 전까지만 매칭합니다.
(d): '수강생 <30>명, 조교 <3명>' 
숫자를 찾아 <>로 감쌉니다.
(e): '수강생 <\x01>명, 조교 <\x01명>'
아스키코드 제어 문자로 치환됩니다.

추가질문 i: .+는 문자열 끝에 있는 >까지 최대한 많이 매칭하지만, .+?는 게으른 수량자이기 때문에 개별 태그를 따로 분리니다.

추가질문 ii: (d)는 r" "을 잘 사용하지만, (e)는 일반 문자열을 사용하여 \1이 아스키제어문자로 해석되어 치환되기 때문입니다.


Q3- a, b, c

In [ ]:
import re
from collections import Counter

def clean_post(post: str) -> str:
    # 1. URL 제거
    post = re.sub(r"https?://\S+", " ", post)
    # 2. HTML 태그 제거
    post = re.sub(r"<[^>]+>", "", post)
    # 3. 이메일 및 전화번호 마스킹
    post = re.sub(r"[\w.+-]+@[\w-]+\.[\w.-]+", "[이메일]", post)
    post = re.sub(r"\d{2,4}-\d{3,4}-\d{4}", "[전화]", post)
    # 4. 멘션 및 해시태그 제거
    post = re.sub(r"[@#]\w+", " ", post)
    # 5. 한글 자음/모음 제거
    post = re.sub(r"[ㄱ-ㅣ]+", "", post)
    # 6. 공백 정리 및 strip
    post = re.sub(r"\s+", " ", post).strip()
    return post

def extract_hashtags(post: str) -> list[str]:
    # # 뒤에 오는 한글, 영문, 숫자 추출 (캡처 그룹 사용)
    return re.findall(r"#([가-힣a-zA-Z0-9]+)", post)

def analyze_posts(posts: list[str]) -> dict:
    cleaned_posts = [clean_post(p) for p in posts]
    
    # 해시태그 집계
    all_tags = []
    for p in posts:
        all_tags.extend(extract_hashtags(p))
    hashtag_counts = dict(Counter(all_tags).most_common())
    
    # 마스킹 건수 집계 (re.subn 활용 권장)
    total_masked = 0
    email_pat = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
    phone_pat = re.compile(r"\d{2,4}-\d{3,4}-\d{4}")
    for p in posts:
        _, e_cnt = email_pat.subn("[이메일]", p)
        _, p_cnt = phone_pat.subn("[전화]", p)
        total_masked += (e_cnt + p_cnt)

    return {
        "posts_n": len(posts),
        "avg_length_after_clean": round(sum(len(p) for p in cleaned_posts) / len(posts), 2),
        "hashtag_counts": hashtag_counts,
        "masked_count": total_masked
    }

clean_post(post) 반환값:
'오늘 수업 진짜 재밌었음!! 감사'
'자료:'
'팀플 어디서 모이지 카톡'
'중요: 다음 시험 범위는 1-15강'
'문의는 [이메일] ([전화])로!'
'여러 공백과 줄바꿈이 많은 텍스트'
'진짜 좋다'

analyze_posts(posts) 반환 딕셔너리
{"posts_n": 7, "avg_length_after_clean": 13.57, "hashtag_counts": {"파이썬": 2, "DCCP2026": 1, "팀플": 1, "추천": 1}, "masked_count": 2}



6단계의 순서를 지켜야 하는 이유:
만약 단계 4(멘션 제거)를 단계 3(이메일 마스킹)보다 먼저 수행하게 되면, 이메일 주소(예: mam3b@snu.ac.kr)에 포함된 @snu 부분이 멘션(@\w+)으로 잘못 인식되어 먼저 공백으로 지워지게 됩니다. 이렇게 되면 이메일의 고유한 패턴이 망가져, 정작 다음 단계에서 이메일을 찾아 [이메일]로 안전하게 마스킹할 수 없게 되므로 반드시 주어진 순서를 지켜야 합니다.